1. T-test: For comparing means of two groups with continuous, normally distributed data
2. Chi-square test: For comparing categorical data (like click/no-click distributions)
3. Mann-Whitney U test: For comparing two groups when data isn't normally distributed
4. ANOVA: For comparing means of three or more groups

Each function:
- Creates sample data appropriate for the test type
- Runs the statistical test
- Prints the test statistic and p-value

To interpret the results:
- For all tests, if p-value < 0.05 (common threshold), we reject the null hypothesis
- This means there's a statistically significant difference between groups

Note: This is simplified example data. In real A/B testing:
- You'd need to check assumptions for each test
- Consider effect sizes
- Possibly use more sophisticated methods like bootstrapping
- Account for multiple comparisons when needed

In [4]:
import numpy as np
from scipy import stats
from scipy.stats import norm

def bootstrap_ci(data, statistic, num_bootstrap_samples=10000, ci=0.95):
    """
    Generic bootstrap confidence interval calculation
    """
    bootstrap_stats = []
    for _ in range(num_bootstrap_samples):
        bootstrap_sample = np.random.choice(data, size=len(data), replace=True)
        bootstrap_stats.append(statistic(bootstrap_sample))
    
    confidence_interval = np.percentile(bootstrap_stats, [(1-ci)/2 * 100, (1+ci)/2 * 100])
    return confidence_interval

def wilson_score_interval(count, nobs, alpha=0.05):
    """
    Calculate Wilson score interval for a proportion
    
    Parameters:
    count: number of successes
    nobs: number of trials
    alpha: significance level (default 0.05 for 95% CI)
    """
    z = stats.norm.ppf(1 - alpha/2)
    p = count / nobs
    
    denominator = 1 + z**2/nobs
    center = (p + z**2/(2*nobs))/denominator
    spread = z * np.sqrt(p*(1-p)/nobs + z**2/(4*nobs**2))/denominator
    
    return (center - spread, center + spread)

def run_ttest_example():
    """
    T-test example with confidence intervals
    """
    # Simulating conversion rates
    group_a = np.random.normal(0.12, 0.03, 1000)
    group_b = np.random.normal(0.15, 0.03, 1000)
    
    # T-test
    t_stat, p_value = stats.ttest_ind(group_a, group_b)
    
    # Calculate confidence intervals
    ci_a = bootstrap_ci(group_a, np.mean)
    ci_b = bootstrap_ci(group_b, np.mean)
    
    print("\nT-Test Results:")
    print(f"t-statistic: {t_stat:.4f}")
    print(f"p-value: {p_value:.4f}")
    print(f"Group A mean: {np.mean(group_a):.4f}, 95% CI: ({ci_a[0]:.4f}, {ci_a[1]:.4f})")
    print(f"Group B mean: {np.mean(group_b):.4f}, 95% CI: ({ci_b[0]:.4f}, {ci_b[1]:.4f})")

def run_chisquare_example():
    """
    Chi-square example with proportion confidence intervals
    """
    # Simulating click data
    observed = np.array([[180, 820],  # Version A: [clicks, no_clicks]
                        [225, 775]])  # Version B: [clicks, no_clicks]
    
    # Chi-square test
    chi2, p_value, dof, expected = stats.chi2_contingency(observed)
    
    # Calculate proportion confidence intervals using Wilson score interval
    n_a = sum(observed[0])
    n_b = sum(observed[1])
    prop_a = observed[0][0] / n_a
    prop_b = observed[1][0] / n_b
    
    # Wilson score interval
    ci_a = wilson_score_interval(observed[0][0], n_a)
    ci_b = wilson_score_interval(observed[1][0], n_b)
    
    print("\nChi-square Test Results:")
    print(f"chi-square statistic: {chi2:.4f}")
    print(f"p-value: {p_value:.4f}")
    print(f"Group A proportion: {prop_a:.4f}, 95% CI: ({ci_a[0]:.4f}, {ci_a[1]:.4f})")
    print(f"Group B proportion: {prop_b:.4f}, 95% CI: ({ci_b[0]:.4f}, {ci_b[1]:.4f})")

def run_mannwhitney_example():
    """
    Mann-Whitney U test example with bootstrap confidence intervals
    """
    # Simulating time-on-page data
    group_a = np.random.exponential(50, 1000)
    group_b = np.random.exponential(55, 1000)
    
    # Mann-Whitney U test
    stat, p_value = stats.mannwhitneyu(group_a, group_b, alternative='two-sided')
    
    # Calculate confidence intervals for medians
    ci_a = bootstrap_ci(group_a, np.median)
    ci_b = bootstrap_ci(group_b, np.median)
    
    print("\nMann-Whitney U Test Results:")
    print(f"statistic: {stat:.4f}")
    print(f"p-value: {p_value:.4f}")
    print(f"Group A median: {np.median(group_a):.4f}, 95% CI: ({ci_a[0]:.4f}, {ci_a[1]:.4f})")
    print(f"Group B median: {np.median(group_b):.4f}, 95% CI: ({ci_b[0]:.4f}, {ci_b[1]:.4f})")

def run_anova_example():
    """
    ANOVA example with confidence intervals
    """
    # Simulating revenue data
    group_a = np.random.normal(100, 20, 1000)
    group_b = np.random.normal(105, 20, 1000)
    group_c = np.random.normal(110, 20, 1000)
    
    # ANOVA
    f_stat, p_value = stats.f_oneway(group_a, group_b, group_c)
    
    # Calculate confidence intervals
    ci_a = bootstrap_ci(group_a, np.mean)
    ci_b = bootstrap_ci(group_b, np.mean)
    ci_c = bootstrap_ci(group_c, np.mean)
    
    print("\nANOVA Test Results:")
    print(f"F-statistic: {f_stat:.4f}")
    print(f"p-value: {p_value:.4f}")
    print(f"Group A mean: {np.mean(group_a):.4f}, 95% CI: ({ci_a[0]:.4f}, {ci_a[1]:.4f})")
    print(f"Group B mean: {np.mean(group_b):.4f}, 95% CI: ({ci_b[0]:.4f}, {ci_b[1]:.4f})")
    print(f"Group C mean: {np.mean(group_c):.4f}, 95% CI: ({ci_c[0]:.4f}, {ci_c[1]:.4f})")

if __name__ == "__main__":
    # Set random seed for reproducibility
    np.random.seed(42)
    
    # Run all tests
    run_ttest_example()
    run_chisquare_example()
    run_mannwhitney_example()
    run_anova_example()



T-Test Results:
t-statistic: -23.7888
p-value: 0.0000
Group A mean: 0.1206, 95% CI: (0.1188, 0.1224)
Group B mean: 0.1521, 95% CI: (0.1503, 0.1540)

Chi-square Test Results:
chi-square statistic: 5.9940
p-value: 0.0144
Group A proportion: 0.1800, 95% CI: (0.1574, 0.2050)
Group B proportion: 0.2250, 95% CI: (0.2002, 0.2519)

Mann-Whitney U Test Results:
statistic: 460457.0000
p-value: 0.0022
Group A median: 31.2661, 95% CI: (28.5795, 33.9104)
Group B median: 38.0585, 95% CI: (34.8490, 42.3013)

ANOVA Test Results:
F-statistic: 44.2429
p-value: 0.0000
Group A mean: 100.7379, 95% CI: (99.5205, 101.9954)
Group B mean: 105.6198, 95% CI: (104.3674, 106.8504)
Group C mean: 109.0714, 95% CI: (107.8994, 110.2797)


In [2]:
# Python example of bootstrapping to estimate a confidence interval for a mean

import numpy as np

def bootstrap_mean_ci(data, num_bootstrap_samples=10000, ci=0.95):
    bootstrap_means = []
    for _ in range(num_bootstrap_samples):
        bootstrap_sample = np.random.choice(data, size=len(data), replace=True)
        bootstrap_means.append(np.mean(bootstrap_sample))
    
    confidence_interval = np.percentile(bootstrap_means, [(1-ci)/2 * 100, (1+ci)/2 * 100])
    return np.mean(data), confidence_interval

# Example usage
data = np.random.normal(100, 20, 1000)  # Sample data
mean, (ci_lower, ci_upper) = bootstrap_mean_ci(data)
print(f"Mean: {mean:.2f}")
print(f"95% Confidence Interval: ({ci_lower:.2f}, {ci_upper:.2f})")


Mean: 99.51
95% Confidence Interval: (98.22, 100.81)
